# Analyse and Visualise the Space Race

Day 99 assignment of the 100 Days of Code Python bootcamp.
The data was scraped from nextspaceflight.com and holds the space
missions since 1957.

### Import Statements

In [ ]:
import pandas as pd
import plotly.io as pio

from space_race import analysis, charts
from space_race.cleaning import clean_missions
from space_race.exploration import explore
from space_race.loading import load_missions

### Notebook Presentation

Charts are shown as pictures so they appear the same everywhere.
Use `pio.renderers.default = 'browser'` to get interactive charts.

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format
pio.renderers.default = 'png'

### Load the Data

In [ ]:
df_data = load_missions()

# Preliminary Data Exploration

* What is the shape of `df_data`?
* How many rows and columns does it have?
* What are the column names?
* Are there any NaN values or duplicates?

In [ ]:
facts = explore(df_data)
print(f'Shape: {df_data.shape}')
print(f'Rows: {facts.row_count}, columns: {facts.column_count}')
print(f'Columns: {facts.column_names}')
print(f'Missing values per column: {facts.missing_by_column}')
print(f'Duplicate rows: {facts.duplicate_count}')

In [ ]:
df_data.head()

## Data Cleaning - Check for Missing Values and Duplicates

Remove the junk index columns and the duplicate row, make `Price`
numeric (USD millions, missing stays missing), parse the dates and
add the launching country.

In [ ]:
df_data = clean_missions(df_data)
print(f'Shape after cleaning: {df_data.shape}')
df_data.isna().sum()

In [ ]:
df_data.head()

## Descriptive Statistics

In [ ]:
df_data.describe(include='all')

## Number of Launches per Organisation

Which organisations launched the most, how many of the rockets are
still active, and how did the missions end?

In [ ]:
per_organisation = analysis.launches_per_organisation(df_data)
charts.bar_chart(
    per_organisation.head(15),
    'Launches per organisation (top 15)',
    'Launches',
    'Organisation',
    horizontal=True,
).show()

In [ ]:
rocket_status = analysis.rocket_status_counts(df_data)
charts.pie_chart(rocket_status, 'Rocket status').show()
mission_status = analysis.mission_status_counts(df_data)
charts.pie_chart(mission_status, 'Mission status').show()

## Launch Prices and Money Spent

Only a fifth of the launches have a published price, so these
figures cover the known prices only (USD millions).

In [ ]:
prices = analysis.known_prices(df_data)
print(f'Launches with a known price: {len(prices)} of {len(df_data)}')
charts.histogram(
    prices, 'Distribution of launch prices', 'Price (USD millions)'
).show()

In [ ]:
spent = analysis.money_spent_per_organisation(df_data)
charts.bar_chart(
    spent.head(10),
    'Money spent per organisation',
    'USD millions',
    'Organisation',
    horizontal=True,
).show()

In [ ]:
average = analysis.average_price_per_organisation(df_data)
charts.bar_chart(
    average.head(10),
    'Average price per launch',
    'USD millions',
    'Organisation',
    horizontal=True,
).show()

## Launches per Year and per Month

How did the number of launches change, which month is the most
popular, and did launches become cheaper?

In [ ]:
per_year = analysis.launches_per_year(df_data)
print(f'Most launches in one year: {per_year.idxmax()} ({per_year.max()})')
charts.line_chart(per_year, 'Launches per year', 'Year', 'Launches').show()

In [ ]:
per_month = analysis.launches_per_month(df_data)
print(f'Most popular month: {analysis.most_popular_month(df_data)}')
charts.bar_chart(
    per_month, 'Launches per month', 'Month', 'Launches'
).show()

In [ ]:
price_over_time = analysis.average_price_per_year(df_data)
charts.line_chart(
    price_over_time, 'Average launch price per year', 'Year', 'USD millions'
).show()

## Launches and Failures per Country

Countries are shaded by the number of launches, using ISO 3166 codes.

In [ ]:
per_country = analysis.launches_per_country(df_data)
charts.choropleth(per_country, 'Launches per country').show()

In [ ]:
failed_per_country = analysis.failures_per_country(df_data)
charts.choropleth(
    failed_per_country, 'Failed launches per country'
).show()

## Sunburst Chart: Country, Organisation and Mission Status

In [ ]:
charts.sunburst_chart(
    analysis.sunburst_data(df_data), 'Country, organisation, status'
).show()

## Organisations and Countries over Time

Which organisation launched the most in each year, and how did the
USA and the USSR compare during the Space Race (1957 to 1991)?

In [ ]:
top_per_year = analysis.top_organisation_per_year(df_data)
charts.grouped_bar_chart(
    top_per_year,
    'Year',
    'Organisation',
    'Organisation with most launches per year',
    stacked=True,
).show()

In [ ]:
cold_war = analysis.cold_war_launches(df_data)
print(cold_war.groupby('Side')['Launches'].sum())
charts.grouped_bar_chart(
    cold_war, 'Year', 'Side', 'USA against USSR: launches per year'
).show()

In [ ]:
cold_war_failures = analysis.cold_war_launches(df_data, failures_only=True)
charts.grouped_bar_chart(
    cold_war_failures, 'Year', 'Side', 'USA against USSR: failures per year'
).show()

## Have Missions Become Safer?

The failure rate is the share of launches that did not end in Success.

In [ ]:
failure_rate = analysis.failure_rate_per_year(df_data)
charts.line_chart(
    failure_rate, 'Failure rate per year', 'Year', 'Share of launches'
).show()
analysis.failure_rate_per_decade(df_data)

**Answer:** yes. About two thirds of the launches of the 1950s did not
succeed, a fifth in the 1960s, and since the 1970s fewer than one in
ten fail.